In [ ]:
import numpy as np
import pandas as pd
import scipy as scp
import matplotlib.pyplot as plt
import matplotlib.cm as cmx
import mne
from mne.datasets import sample
import os.path as op
import os
from tqdm import tqdm
from src.utils import event_shape,init_hmp,GAMMA2,plot_hsmm,event_shape,plot_val_hsmm,sample_hmp,simulate
from mssmViz.plot import *
from mssm.models import *
from mssmViz.plot import *
from mssm.src.python.gamm_solvers import gd_coef_smooth,correct_coef_step_gen_smooth
from mssm.src.python.formula import build_model_matrix
from mssm.src.python.hsmm import HSMMFamily
from mssm.src.python.mcmc import sample_mssm
from mssm.src.python.utils import computeAr1Chol
from hmp.utils import read_mne_data,transform_data
import dotenv
dotenv.load_dotenv()

nc = 10
colors = [cmx.tab20(x) for x in np.linspace(0.0,min(0.1*nc,1.0),nc)]
single_size = 8/2.54

plt.rcParams["font.family"] = "Source Sans 3"
plt.rcParams["font.weight"] = "semibold"
plt.rcParams["font.size"] = 9
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["figure.titlesize"] = 12
math_font_size = 8
plt.rcParams['mathtext.fontset'] = 'cm'

results_path = os.getenv("results_path")
use_cores = int(os.getenv("n_cores"))

In [ ]:
# Code (and comments) below was largely taken from the HMP tutorials available on Github to
# simulate data for a HMP model and to load that data back in. Code has been adapted slightly to
# simulate a non-linear effect of covariate x on the duration of flat states.
# see: https://github.com/GWeindel/hmp/blob/devel/docs/source/notebooks/1-How_HMP_works.ipynb

cpus = 4 # For multiprocessing 

n_trials = 500 #Number of trials to simulate
sfreq = 100

##### Here we define the sources of the brain activity (event) for each trial
n_events = 3
frequency = 10. #Frequency of the event defining its duration, half-sine of 10Hz = 50ms
amplitude = .2e-7 #Amplitude of the event in nAm, defining signal to noise ratio
scale = 0.5 # Scale parameter of Gamma distribution when expressed as member of exponential family

#Which source to activate for each event (see atlas when calling simulations.available_sources())
names = ['inferiortemporal-lh','caudalanteriorcingulate-rh','bankssts-lh','superiorparietal-lh']


In [ ]:
# Generate sources
sources = []
true_durs = np.zeros((n_events+1,n_trials))

# Simulate effect of x on expected duration of each stage
np_gen = np.random.default_rng(2025)

x = np_gen.random(size=n_trials)
gam_dat = pd.DataFrame({"x":x})

#One source = one frequency, one amplitude and a given by-trial variability distribution
for si,name in enumerate(names): 

    # Get mean of stage-specific Gamma distribution. Functions from Gu & Whaba (1991)
    # "Minimizing GCV/GML Scores with Multiple Smoothing Parameters via the Newton Method"
    if si == 0:
        fv = 0.5 * np.sin(np.pi*x) + 5
    elif si == 1:
        fv = 0.05*np.exp(2*x) + 5
    elif si == 2:
        fv = (0.2*np.power(x,11)*np.power(10*(1-x),6)+10*np.power(10*x,3)*np.power(1-x,10))*0.2 + 5
    else:
        fv = np.zeros_like(x) + 5
    
    # Plot functi
    plt.scatter(x,np.exp(fv))
    plt.show()
    mu = Gamma().link.fi(fv)

    # Convert to parameters scipy expects
    alpha = 1/scale
    beta = alpha/mu

    # duration samples  
    dur = scp.stats.gamma.rvs(a=alpha,scale=(1/beta),size=n_trials,random_state=2025)

    dur[dur < 50] = 70
    
    # Collect durations
    true_durs[si,:] = dur
    gam_dat[f"durs_{si}"] = dur
    
    sources.append([name, frequency, amplitude, dur])

# Function used to generate the data
file = simulate(sources, n_trials, cpus, 'dataset_raw',
                 overwrite=False, sfreq=sfreq, seed=None,path="./")

#load electrode position, specific to the simulations
data_path = sample.data_path()
subjects_dir = op.join(data_path, 'subjects')
subject = 'sample'
evoked_fname = op.join(data_path, 'MEG', subject, 'sample_audvis-ave.fif')
info = mne.io.read_info(evoked_fname, verbose=False)
positions = np.delete(mne.channels.layout._find_topomap_coords(info, 'eeg'),52,axis=0)

In [ ]:
# Recovering the events to epoch the data (in the number of trials defined above)
events = np.load(file[1])
resp_trigger = int(np.max(np.unique(events[:,2]))) # Resp trigger is the last source in each trial

# Keeping only stimulus and response triggersfile
event_id = {'stimulus':1} # trigger 1 = stimulus
resp_id = {'response':resp_trigger}


In [ ]:
raw = mne.io.read_raw_fif(file[0], preload=False, verbose=False)

# Reading the data
eeg_data = read_mne_data(file[0], event_id=event_id, resp_id=resp_id, sfreq=sfreq, 
            events_provided=events, verbose=False)
eeg_data
hmp_data = transform_data(eeg_data, apply_standard=True,n_comp=3)

In [ ]:
# Now we get the data into the format required by mssm.

# Prepare data for mssm
pd_dat = hmp_data.to_dataframe("pc")
pd_dat["component"] = pd_dat.index.get_level_values(0)
pd_dat = pd_dat.droplevel(["epochs","samples","participant"])
pd_dat = pd_dat.reset_index(drop=True)
pd_dat["component"] = "pc_" + pd_dat["component"].astype(str)

pd_dat = pd_dat.sort_values(['epochs', 'component' ,'samples'], ascending=[True,True,True])
pd_dat = pd_dat.reset_index(drop=True)

pd_dat.head()

# Get into wide format
dat = pd_dat.iloc[pd_dat["component"].values == "pc_0",:].copy()
dat["pc0"] = dat["pc"].values
dat = dat.drop(columns=["component","pc"])
dat = dat.reset_index(drop=True)

for pc in range(1,len(np.unique(pd_dat["component"]))):
    dat[f"pc{pc}"] = pd_dat.loc[pd_dat["component"].values == f"pc_{pc}","pc"].values

dat["x"] = 5.0
for epoch in np.unique(dat["epochs"]):
    dat.loc[dat["epochs"].values == epoch,"x"] = x[epoch]

# Extract rts and x values for trials not dropped
rts = true_durs.sum(axis=0)[np.unique(np.array(dat["epochs"]))]
xts = x[np.unique(np.array(dat["epochs"]))]

print(dat.head())

In [ ]:
# Split vector per series/epoch
uval, id = np.unique(np.array(dat["epochs"]),return_index=True)
sid = np.sort(id)
split_series = sid[1:]

# Rts in samples - multiply by 10 to get durations back in ms.
rtss = []
rtss.extend(sid[1:] - sid[0:-1])
rtss.append(dat.shape[0] - sid[-1])

print(max(rtss),min(rtss))

In [ ]:
# Set up template
template = event_shape(100,50)

# Set up formulas
o_family = None # HMP template/bump is assumed observation model
d_family = GAMMA2(links=[LOG()]) # Gamma 2 for stage durations 
n_S = 7
M = 3
D = max(rtss)
tid = np.arange(len(xts))

ys = []
Xs = []
form_n_coef = []
init_coef = []
obs_families = []
shared_formulas = []
obs_formulas = []
build_matrix = []
build_mat_idx = []
links = []
pars = 0


ar=False
rho = 0.5
Lrhoi = None

if ar:
    ar_form = Formula(lhs(f"pc{1}"),[i()],data=dat,series_id="epochs")
    Lrhoi,_ = computeAr1Chol(ar_form,rho)

# Emission models
event = 0
start_sep = len(build_matrix)
for j in range(n_S):
    obs_families.append([])

    for m in range(M):

        obs_families[-1].append(o_family)

        if j % 2 == 1:
            
            # Model of mean of obs model in state j and of signal m
            obs_formulas.append(Formula(lhs(f"pc{m}"),[i()],data=dat))

            init_coef.extend(scp.stats.norm.rvs(size=obs_formulas[-1].n_coef,
                                                random_state=event*(m+1) + m))
            form_n_coef.append(obs_formulas[-1].n_coef)
            _ = build_penalties(obs_formulas[-1])
            ys.append(dat[f"pc{m}"].values.reshape(-1,1))

            if len(build_matrix) == start_sep:
                build_matrix.append(True)
                Xs.append(build_model_matrix(obs_formulas[-1]))
            else:
                build_matrix.append(False)
                Xs.append(None)
            build_mat_idx.append(start_sep)
        
            links.extend([Identity()])
            if m >= (M-1):
                event += 1
            pars += 1


# Duration models
d_dat = pd.DataFrame({"x":xts,
                      "d":np.arange(len(xts)),
                      "sub":np.ones(len(xts))})

d_dat = d_dat.astype({"sub":"O"})

d_families = []
d_formulas = []

flat = 0
start_durs = len(build_mat_idx)
for j in range(n_S):

    if j % 2 == 1:
        d_families.append(None)
    else:

        d_families.append(d_family)

        # Model of mean of dur model in state j
        # Note, ``ri("sub")`` here actually replaces an intercept, since the sampler code
        # needs at least one penalized set of coefficients.
        d_formulas.append(Formula(lhs("d"),[ri("sub")],data=d_dat)) 
        if flat == 0:
            build_matrix.append(True)
            _ = build_penalties(d_formulas[-1])
            Xs.append(build_model_matrix(d_formulas[-1]))
        else:
            build_matrix.append(False)
            Xs.append(None)
        
        build_mat_idx.append(start_durs)
        form_n_coef.append(d_formulas[-1].n_coef)
        init_coef.extend([5])
        ys.append(d_dat["d"].values.reshape(-1,1))

        links.extend(d_family.links)
        flat += 1
        pars += 1

# Initialize Family
T = np.diag(
    np.ones(n_S - 1), 1
)  # Super diagonal shift matrix for transitions

pi = np.zeros(n_S)  # Start in first state
pi[0] = 1

fam = HSMMFamily(pars,
                links,
                n_S,
                obs_fams=obs_families,
                d_fams=d_families,
                sid = sid,
                tid = tid,
                D = D,
                M = M,
                starts_with_first=True,
                ends_with_last=True,
                ends_in_last=True,
                T=T,
                pi=pi,
                scale = 1,
                event_template=template,
                hmp_fam="Gaussian",
                n_cores=4,
                build_mat_idx=build_mat_idx,
                Lrhoi=Lrhoi)

# Initialize coef
coef_split_idx = form_n_coef[:-1]

for coef_i in range(1,len(coef_split_idx)):
    coef_split_idx[coef_i] += coef_split_idx[coef_i-1]

total_coef = np.sum(form_n_coef)
init_coef = np.array(init_coef).reshape(-1,1)

# Set up pseudo-penalty that is all zero
S_emb = scp.sparse.csc_array((total_coef,total_coef))


In [ ]:
# Estimate coefficients via gradient descent - model is simple enough for that
conv_tol = 1e-7
a = 0.01
coef = copy.deepcopy(init_coef)
c_llk = fam.llk(coef,coef_split_idx,ys,Xs)
llk_warp = lambda x: fam.llk(x.reshape(-1,1),coef_split_idx,ys,Xs)

for oi in tqdm(range(100)):

    grad = fam.gradient(coef,coef_split_idx,ys,Xs)
    #grad = scp.optimize.approx_fprime(coef.flatten(),llk_warp)
    #grad = grad.reshape(-1,1)
    next_coef = gd_coef_smooth(coef,grad,S_emb,a)
    #print(coef)

    # Prepare to check convergence
    prev_llk_cur_pen = c_llk - 0.5*coef.T@S_emb@coef

    # Perform step length control
    # Steps might throw warnings, return nan or inf,
    # but mssm handles that.
    with warnings.catch_warnings(): 
        warnings.simplefilter("ignore")
        coef,c_llk,c_pen_llk,a = correct_coef_step_gen_smooth(fam,ys,Xs,coef,next_coef,
                                                              coef_split_idx,c_llk,S_emb,a)

    if np.abs(c_pen_llk - prev_llk_cur_pen) < conv_tol*np.abs(c_pen_llk):
        break

print(coef)

In [ ]:
# Plot an overview of the HMP estimate
fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

plot_hsmm([fam,coef,coef_split_idx,Xs],ys,d_dat,
          network_graph=False,n_samples=10,
          series=3,fig=fig,plot_state_labels=False,
          magnitudes=[coef[bump*M:(bump+1)*M] for bump in range(3)],
          pc_weights=hmp_data.attrs['pca_weights'].data,
          positions=positions,topo_width=17.5,topo_height_rel=3.75,scale=0.1
          )
plt.savefig(results_path + "hmp_overview.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Plot the residuals of the HMP - they generally look fine. The auto-correlation can
# largely be adressed by setting ar = True in the model specification above.
# However, we know that the model of the expected flat state durations is wrong in this case -
# but as mentioned in the paper, this really does not show up in the forward pseudo residuals.
fig = plt.figure(figsize=(19 / 2.54, 6 / 2.54), layout="constrained")

plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,0,
              per_component=False,
              seed=1,predictive_examples=500,
              quantile_label_pos=0.65,wspace=0.01,fig=fig,n_samples=100,
              select_component=0
             )

plt.savefig(results_path + "resid_overview.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# As an alternative we look at the average durations as defined in the paper - these do show that
# the residuals depend strongly on covariate x - and, for flat states 1 and 3, we can actually recover
# the true function of x quite accurately from the residuals.
# So we need to adjust the model of the expected flat state durations.
fig = plt.figure(figsize=(19 / 2.54, 19 / 2.54), layout="constrained")
plot_val_hsmm([fam,coef,coef_split_idx,Xs],ys,dat,d_dat,4,
              per_component=False,fig=fig,
              seed=2,predictive_examples=5,
              quantile_label_pos=0.6,wspace=0.01,
              pred_vars=["x"],resid_states=[0,2,4,6],
              n_samples=100,resid_terms=[i(),f(["x"])],
              type_3_res="posterior_dur",hspace=0.05,plot_partial=None)

plt.savefig(results_path + "resid_overview2.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
# We first decode the viterbi path of the simpler model to initialize the parameters of the more
# complex model
viterbi = fam.decode_viterbi(coef,coef_split_idx,ys,Xs)

In [ ]:
# Set up formulas
ys = []
Xs = []
form_n_coef = []

obs_families = []
obs_formulas = []
build_mat_idx = []
build_matrix = []
links = []
pars = 0

# Emission models

event = 0
start_sep = 0
for j in range(n_S):
    obs_families.append([])

    for m in range(M):

        obs_families[-1].append(o_family)

        if j % 2 == 1:
            
            # Model of mean of obs model in state j and of signal m
            obs_formulas.append(Formula(lhs(f"pc{m}"),[i()],data=dat)) 
            form_n_coef.append(obs_formulas[-1].n_coef)
            _ = build_penalties(obs_formulas[-1])
            ys.append(dat[f"pc{m}"].values.reshape(-1,1))

            if len(build_matrix) == start_sep:
                build_matrix.append(True)
                Xs.append(build_model_matrix(obs_formulas[-1]))
            else:
                build_matrix.append(False)
                Xs.append(None)
            build_mat_idx.append(start_sep)

            links.extend([Identity()])
            if m >= (M-1):
                event += 1
            pars += 1

# Duration models

d_dat = pd.DataFrame({"x":xts,
                      "d":np.arange(len(xts))})

d_families = []
d_formulas = []

flat = 0
start_d = len(build_mat_idx)
for j in range(n_S):

    if j % 2 == 1:
        d_families.append(None)
    else:

        d_families.append(d_family)

        # Model of mean of dur model in state j - now includes a function of covariate x
        d_formulas.append(Formula(lhs("d"),[i(),f(["x"])],data=d_dat)) 

        if flat == 0:
            build_matrix.append(True)
            _ = build_penalties(d_formulas[-1])
            Xs.append(build_model_matrix(d_formulas[-1]))
        else:
            build_matrix.append(False)
            Xs.append(None)
        
        build_mat_idx.append(start_d)
        form_n_coef.append(d_formulas[-1].n_coef)
        ys.append(d_dat["d"].values.reshape(-1,1))


        links.extend(d_family.links)
        flat += 1
        pars += 1

# Initialize Family
fam2 = HSMMFamily(pars,
                  links,
                  n_S,
                  obs_fams=obs_families,
                  d_fams=d_families,
                  sid = sid,
                  tid = tid,
                  D = D,
                  M = M,
                  starts_with_first=True,
                  ends_with_last=True,
                  ends_in_last=True,
                  T=T,
                  pi=pi,
                  scale = 1,
                  event_template=template,
                  hmp_fam="Gaussian",
                  n_cores=use_cores,
                  build_mat_idx=build_mat_idx,
                  Lrhoi=Lrhoi)

coef_split_idx = form_n_coef[:-1]

for coef_i in range(1,len(coef_split_idx)):
    coef_split_idx[coef_i] += coef_split_idx[coef_i-1]

total_coef = np.sum(form_n_coef)


In [ ]:
# We can initialize the parameters (coefficients and regularization parameters for the smooth terms
# of covariate x) of this updated model from the viterbi-decoed state estimates
# of the simpler hmp model.
init_coef, init_lambda = init_hmp(dat,d_dat,obs_formulas,d_formulas,
                                  [viterbi[s][1] for s in range(len(viterbi))],
                                  fam2,Xs,ys,True,None,should_plot=True)

In [ ]:
# One option to estimate this model is the Empirical Bayes approach discussed in the paper.
# This can be achieved via mssm:
hsmm = GSMM([*obs_formulas,*d_formulas],family=fam2)
hsmm.fit(method="qEFS",init_coef=init_coef,
         max_restarts=-1,seed=5,control_lambda=1,
         global_opt_qefs=True,init_lambda=init_lambda)

In [ ]:
# mssmViz has plotting functions for GSMM models, which we can use to visualize the estimated
# effect of covairate x
plot(hsmm,dist_par=11)

In [ ]:
# mssm also enables fully Bayesian estimation for any GSMM model:
llks, coef_samples, rho_samples = sample_mssm(hsmm,
                                              auto_converge=False,
                                              M_adapt=20,
                                              parallelize_chains=False,
                                              n_chains=1,
                                              sample_rho=False,
                                              delta=0.6,
                                              n_iter=100,
                                            )

In [ ]:
# Trace plots
plt.plot(coef_samples[0,:,0])
plt.show()

In [ ]:
# A final alternative discussed in the paper is the Laplace approximate sampling approach, which has
# been implemented for hmp models in the function sample_hmp
coef_samples2,lam_samples, hsmm2 = sample_hmp(fam2,
                                            [init_coef + scp.stats.norm.rvs(size=len(init_coef)).reshape(-1,1) for _ in range(2)],
                                            coef_split_idx,[*obs_formulas,*d_formulas],
                                            Xs,ys,
                                            n_burnin=0,
                                            n_iter=100,
                                            callback=None,
                                            n_chains=2)

In [ ]:
# We just take the coefficients from the Laplace approximate approach:
coef = np.mean(coef_samples2,axis=(0,1)).reshape(-1,1)

In [ ]:
plot(hsmm2,dist_par=11)

In [ ]:
# We can now look at the updated average duration residuals, which have improved substantially:
fig = plt.figure(figsize=(19 / 2.54, 19 / 2.54), layout="constrained")
plot_val_hsmm([fam2,coef,coef_split_idx,Xs],ys,dat,d_dat,4,
              per_component=False,fig=fig,
              seed=2,predictive_examples=5,
              quantile_label_pos=0.6,wspace=0.01,
              pred_vars=["x"],resid_states=[0,2,4,6],
              n_samples=100,resid_terms=[i(),f(["x"])],
              type_3_res="posterior_dur",hspace=0.05,plot_partial=None)

plt.savefig(results_path + "resid_overview3.pdf", format="pdf", bbox_inches='tight')
plt.show()